# 01 — Exploratory Data Analysis & Data Quality

**Objective**: Perform publication-quality exploratory analysis and rigorous data-quality assessment of the Pima Indians Diabetes Database.

**Clinical context**: Understanding distributions, missingness patterns, and class imbalance is a prerequisite for trustworthy risk-prediction models.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

from src.data.loader import load_pima_dataset
from src.data.quality import assess_data_quality, print_quality_summary
from src.utils.reproducibility import set_seed

set_seed(42)
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 300

In [ ]:
df = load_pima_dataset()
df.head()

In [ ]:
report = assess_data_quality(df, target="Outcome")
print_quality_summary(report)

## Target distribution

Approximately 35 % of patients have a positive diabetes label. This moderate imbalance motivates the use of class weighting or resampling and prioritization of PR-AUC and sensitivity alongside ROC-AUC.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.5))
df["Outcome"].value_counts().sort_index().plot(kind="bar", ax=ax, color=["#4C72B0", "#C44E52"])
ax.set_xticklabels(["No Diabetes", "Diabetes"], rotation=0)
ax.set_ylabel("Count")
ax.set_title("Outcome Distribution — Pima Indians Diabetes Database")
plt.tight_layout()
plt.savefig("../reports/figures/outcome_distribution.png", bbox_inches="tight")
plt.show()

## Feature distributions by outcome

Glucose and BMI show the clearest separation between outcome classes, consistent with known pathophysiology of type 2 diabetes.

In [ ]:
features = ["Glucose", "BMI", "Age", "Insulin", "BloodPressure", "DiabetesPedigreeFunction"]
fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for ax, col in zip(axes.ravel(), features):
    for outcome, color in [(0, "#4C72B0"), (1, "#C44E52")]:
        subset = df.loc[df["Outcome"] == outcome, col].dropna()
        ax.hist(subset, bins=25, alpha=0.55, color=color, label=f"Outcome={outcome}", density=True)
    ax.set_title(col)
    ax.legend(fontsize=7)
plt.suptitle("Feature Distributions by Outcome", y=1.02)
plt.tight_layout()
plt.savefig("../reports/figures/feature_distributions.png", bbox_inches="tight")
plt.show()

## Correlation heatmap

Moderate correlations exist among metabolic features (Glucose–Insulin, BMI–SkinThickness). No pair exceeds the conventional multicollinearity threshold of |r| ≥ 0.9.

In [ ]:
num_cols = ["Pregnancies", "Glucose", "BloodPressure", "SkinThickness",
            "Insulin", "BMI", "DiabetesPedigreeFunction", "Age", "Outcome"]
corr = df[num_cols].corr()
fig, ax = plt.subplots(figsize=(8, 6.5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            square=True, ax=ax, annot_kws={"size": 8})
ax.set_title("Pearson Correlation Heatmap")
plt.tight_layout()
plt.savefig("../reports/figures/correlation_heatmap.png", bbox_inches="tight")
plt.show()

## Clinical interpretation of EDA findings

1. **Glucose** is the strongest univariate discriminator — expected given its central role in diagnostic criteria.
2. **BMI** reflects adiposity-driven insulin resistance.
3. **Age** shows a rightward shift among positive cases, consistent with increasing incidence after mid-life.
4. Missingness (zeros) is concentrated in Insulin and SkinThickness; median imputation on the training fold only is appropriate.
5. Class imbalance is moderate; class weighting is preferred over aggressive oversampling for this sample size.